In [0]:
dbutils.library.restartPython()

In [0]:
import sys
import os
# import importlib
# import utility.deduplication
# importlib.reload(utility.deduplication)
from datetime               import datetime, timezone
from pyspark.sql            import functions as F
sys.path.insert(0, "/Workspace/Users/jeremi.santoso@metrodata.co.id/test_btn/")

from config.config_load     import load_config
from utils.logger           import get_logger, log_job
from utils.deduplication    import deduplicate
from utils.silver_write     import merge_silver
from utils.standardization  import std_category, std_date, std_numeric_only, std_allowed_chars

In [0]:
# JOB CONFIGURATION -----------
# TABLE_NAME = "customer"
dbutils.widgets.text("table_name", "customer")   # POP UP mengisi nama table
TABLE_NAME = dbutils.widgets.get("table_name")
JOB_NAME   = f"silver_{TABLE_NAME}"

TABLE_CONFIG_PATH = f"/Workspace/Users/jeremi.santoso@metrodata.co.id/test_btn/config/table_{TABLE_NAME}.yaml"
# LOGGER -----------
logger = get_logger(JOB_NAME)
logger.info("Table Configuration loaded successfully")

# LOAD JOB AND TABLE CONFIGURATION -------------
logger.info("Loading configuration")
cfg = load_config(TABLE_CONFIG_PATH)

table = cfg["tables"][TABLE_NAME]

catalog       = cfg["catalog"]
source_schema = cfg["schemas"]["bronze"]
target_schema = cfg["schemas"]["silver"]

bronze_table  = f"{catalog}.{source_schema}.{table['bronze_table']}"
silver_table  = f"{catalog}.{target_schema}.{table['silver_table']}"
log_table  = f"{catalog}.{target_schema}.log_jobs"

primary_keys     = table["silver"]["primary_keys"]
incremental_cols = table["silver"]["incremental_columns"]
partition_cols   = table["silver"].get("partition_columns", None)
dedup_cols       = table["silver"].get("deduplicate", True)

## Define Column Cleansing
cleansing_cols = {
    c["name"]: c["null_markers"]
    for c in table["cleansing"]["columns"]
}

logger.info("Schema and Table Configuration loaded successfully")


In [0]:
# TRANSFORMATION ----------
def transform(df):
    #DEFINE VARIABEL untuk STD dari file table yaml
    for column_def in table["columns"]:
        col_name  = column_def["name"]
        col_type = column_def.get("type")
        cleansing = column_def.get("cleansing", {})
        src_name = column_def.get("source", col_name)

    #RENAME NAMA KOLOM DARI BRONZE
        if src_name != col_name:
            if src_name not in df.columns:
                continue  # kolom sumber tidak ada, skip
            df = df.withColumnRenamed(src_name, col_name)            
        elif col_name not in df.columns:
            continue
    
    #HANDLING VALUE MENJADI NULL
    null_markers = cleansing.get("null_markers")
    if null_markers:
        df = df.withColumn(
            col_name,
            F.when(F.col(col_name).isin(null_markers), None).otherwise(F.col(col_name))
    )

    #TRIMMING VALUE
        if cleansing.get("trim"):
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    #CASTING TIPE DATA
        if col_type:
            df = df.withColumn(col_name, F.col(col_name).cast(col_type))

    #STANDARISASI REPLACE VALUE
        mapping = cleansing.get("mapping")
        if mapping:
            df = std_category(
                df,
                column_name  = col_name,
                mapping = mapping,
                default = cleansing.get("default")
            )
    
    #STANDARISASI SPESIAL KARAKTER
        allowed_cfg = cleansing.get("special_char")
        if allowed_cfg:
            df = std_allowed_chars(
                df, col_name,
                condition_column = allowed_cfg["lookup_column"],
                rules             = allowed_cfg["rules"]
            )

    #STANDARISASI KOLOM NUMERIC ONLY
        strip_cfg = cleansing.get("numeric_only")
        if strip_cfg:
            if isinstance(strip_cfg, dict):
                df = std_numeric_only(
                    df,
                    col_name,
                    lookup_column = strip_cfg.get("lookup_column"),
                    values = strip_cfg.get("values")
                )
            else:
                df = std_numeric_only(df, col_name)

    #STD CONCAT DATE+TIME
    df = std_date(df,table)  
    #     df = df.withColumn(
    #     "CFTIMESTAMP",
    #     F.to_timestamp(
    #         F.concat(
    #             F.lpad(F.col("CFVDT6").cast("string"), 6, "0"),
    #             F.lpad(F.col("CFVTME").cast("string"), 6, "0")
    #         ),
    #         "ddMMyyHHmmss"
    #     )
    # )

    return df

In [0]:
df_test = spark.read.table(bronze_table)
df_result = transform(df_test)

display(df_result)
# df_result.select("CFCIF#","CFCLAS","CFNA1").filter(F.col("CFCIF#")=="A219821").display()

In [0]:
def main():
    RUN_ID     = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
    START_TIME = datetime.now(timezone.utc)
    logger.info(f"JOB STARTED | RUN_ID={RUN_ID} | TABLE={TABLE_NAME}")

    try:
        # READ BRONZE -------
        logger.info(f"Reading Bronze: {bronze_table}")
        df = spark.read.table(bronze_table)
        logger.info(f"Bronze row count: {df.count()}")

        # RUN TRANSFORM FUNCTION ----------
        logger.info("Running transform & cleansing")
        df_transform  = transform(df)
        logger.info(f"Row count after transform: {df_transform.count()}")

        # DEDUP ------
        if dedup_cols:
            logger.info("Running deduplication")
            df_final = deduplicate(
                df_transform,
                primary_keys = primary_keys,
                incremental_cols = incremental_cols
            )
            logger.info(f"Row count after dedup: {df_final.count()}")
        else:
            df_final = df_transform
            logger.info("Deduplication skipped")

        row_count = df_final.count()

        # ── WRITE SILVER ──────────────────────────────────────────────────────
        logger.info(f"Merging into Silver: {silver_table}")
        merge_silver(
            spark        = spark,
            df           = df_final,
            table_name   = silver_table,
            primary_keys = primary_keys,
            partition_cols = partition_cols,
        )
        logger.info("Merge complete")

        # DONE
        END_TIME = datetime.now(timezone.utc)
        elapsed = round((datetime.now(timezone.utc) - START_TIME).total_seconds(), 2)
        
        # WRITE STATUS TO LOG TABLE ----------
        log_job(
            spark      = spark,
            dbutils    = dbutils,
            table_name = log_table,
            run_id     = RUN_ID,
            job_name   = JOB_NAME,
            status     = "SUCCESS",
            start_time = START_TIME,
            end_time   = END_TIME,
            duration   = elapsed,
            row_count  = row_count,
        )

        logger.info(f"JOB FINISHED | RUN_ID={RUN_ID} | elapsed={elapsed}s")

    except Exception as e:
        # WRITE FAILED TO LOG TABLE
        END_TIME = datetime.now(timezone.utc)
        elapsed = round((datetime.now(timezone.utc) - START_TIME).total_seconds(), 2)
        logger.error(f"JOB FAILED | RUN_ID={RUN_ID} | elapsed={elapsed}s")
        logger.error(str(e))

        log_job(
            spark      = spark,
            dbutils    = dbutils,
            table_name = log_table,
            run_id     = RUN_ID,
            job_name   = JOB_NAME,
            status     = "FAILED",
            start_time = START_TIME,
            end_time   = END_TIME,
            duration   = elapsed,
            row_count  = None,
            message    = str(e),
        )

        raise

# COMMAND ----------
main()